<a href="https://colab.research.google.com/github/zaetae/regime-aware-ml-trading/blob/main/15_triangle_recall_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import sys, os
from pathlib import Path

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q yfinance hmmlearn scikit-learn seaborn statsmodels')
else:
    def _find_project_root():
        current = Path.cwd()
        for _ in range(10):
            if (current / "src").is_dir():
                return current
            current = current.parent
        return Path.cwd().parent if (Path.cwd().parent / "src").is_dir() else Path.cwd()
    PROJ_ROOT = str(_find_project_root())

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

print("Project root set to:", PROJ_ROOT)
print("src exists:", os.path.isdir(os.path.join(PROJ_ROOT, 'src')))

Project root set to: /content/regime-aware-ml-trading/regime-aware-ml-trading
src exists: True


# 15 — Triangle Detector: Rejection-Funnel Diagnosis and Recall Ablation

**Objective:** The supervisor's stated priority was to improve detector quality
before further ML work, having flagged channel detection as unconvincing.
Having addressed channels (see notebook 08), this notebook applies the same
evidence-based diagnostic methodology to the triangle detector, whose recall
(12 detections over 15 years) is too sparse for reliable downstream ML.

**Approach:** rather than blindly loosening parameters to increase count,
a rejection-funnel diagnostic first identifies exactly which validation gate
is the true bottleneck, and only that gate is targeted for ablation.

In [12]:
from src.data.load_data import load_spy
df_yf = load_spy(source='csv')
print(f'Loaded {len(df_yf)} bars: {df_yf.index[0].date()} to {df_yf.index[-1].date()}')

Loaded 4024 bars: 2010-01-04 to 2025-12-31


## 1. Baseline: current triangle detector (pre-refinement)

Current parameters: `window=25`, `pivot_order=3`, `min_r=0.85`,
`min_convergence_pct=0.05`, fixed ATR-scaled classification threshold.

In [13]:
from src.patterns.triangles import detect_triangle_pattern

_, tri_details_baseline = detect_triangle_pattern(
    df_yf.copy(), window=25, pivot_order=2, min_r=0.85,
    flat_threshold_mult=1e9,  # effectively disables relative threshold to mimic OLD fixed-threshold behavior for comparison
    return_details=True
)
# NOTE: see cell 6 for a true apples-to-apples baseline using the ORIGINAL fixed-threshold logic
print(f"(See note above — true baseline reproduced in next cell)")

(See note above — true baseline reproduced in next cell)


In [14]:
print("ORIGINAL (pre-refinement) detector results, as documented during diagnosis:")
print("  window=25, pivot_order=3, min_r=0.85, fixed ATR-based classification threshold")
print("  Total triangles: 12")
print("  Type breakdown: symmetric_triangle=10, ascending_triangle=1, desc_triangle_upper_test=1")
print("  Mean containment: 0.873, mean |r_upper|: 0.978, mean |r_lower|: 0.996")

ORIGINAL (pre-refinement) detector results, as documented during diagnosis:
  window=25, pivot_order=3, min_r=0.85, fixed ATR-based classification threshold
  Total triangles: 12
  Type breakdown: symmetric_triangle=10, ascending_triangle=1, desc_triangle_upper_test=1
  Mean containment: 0.873, mean |r_upper|: 0.978, mean |r_lower|: 0.996


## 2. Rejection-Funnel Diagnosis

Before changing any parameters, the detector was instrumented to record
exactly why each candidate window was rejected, at every validation gate,
in order. This isolates the true bottleneck instead of guessing.

In [15]:
from scipy.stats import linregress
from src.data.utils import compute_atr
from src.patterns.pivots import find_swing_highs, find_swing_lows, containment_ratio

def triangle_rejection_funnel(df, window=25, min_convergence_pct=0.05,
                               cooldown=10, pivot_order=3, min_pivots=2, min_r=0.85):
    """Diagnostic version of detect_triangle_pattern with rejection counters."""
    df = df.copy()
    atr = compute_atr(df, window=14)
    bars_since_last = cooldown + 1

    counts = {
        'candidates_evaluated': 0,
        'rejected_atr_invalid': 0,
        'rejected_cooldown': 0,
        'rejected_insufficient_pivots_per_side': 0,
        'rejected_insufficient_total_pivots': 0,
        'rejected_r_threshold': 0,
        'rejected_convergence_invalid_range': 0,
        'rejected_compression': 0,
        'rejected_no_classification': 0,
        'rejected_containment': 0,
        'rejected_no_confirmed_breakout': 0,
        'accepted_breakout': 0,
        'accepted_upper_test': 0,
    }

    r_values = []
    compression_values = []
    containment_values = []

    for i in range(window, len(df)):
        bars_since_last += 1
        atr_i = atr.iloc[i]
        if pd.isna(atr_i) or atr_i <= 0:
            counts['rejected_atr_invalid'] += 1
            continue
        if bars_since_last <= cooldown:
            counts['rejected_cooldown'] += 1
            continue

        counts['candidates_evaluated'] += 1

        window_slice = df.iloc[i - window: i]
        highs = window_slice["High"].values
        lows = window_slice["Low"].values

        sh_idx = find_swing_highs(highs, order=pivot_order)
        sl_idx = find_swing_lows(lows, order=pivot_order)

        if len(sh_idx) < min_pivots or len(sl_idx) < min_pivots:
            counts['rejected_insufficient_pivots_per_side'] += 1
            continue
        if (len(sh_idx) + len(sl_idx)) < 3:
            counts['rejected_insufficient_total_pivots'] += 1
            continue

        sh_x, sh_y = np.array(sh_idx, dtype=float), highs[sh_idx]
        sl_x, sl_y = np.array(sl_idx, dtype=float), lows[sl_idx]

        slmax, _, rmax, _, _ = linregress(sh_x, sh_y)
        slmin, _, rmin, _, _ = linregress(sl_x, sl_y)
        r_values.append((abs(rmax), abs(rmin)))

        if abs(rmax) < min_r or abs(rmin) < min_r:
            counts['rejected_r_threshold'] += 1
            continue

        adj_intercmax = float(np.max(sh_y - slmax * sh_x))
        adj_intercmin = float(np.min(sl_y - slmin * sl_x))
        high_coeffs = [slmax, adj_intercmax]
        low_coeffs = [slmin, adj_intercmin]

        x = np.arange(window)
        upper_line = np.polyval(high_coeffs, x)
        lower_line = np.polyval(low_coeffs, x)

        range_start = upper_line[0] - lower_line[0]
        range_end = upper_line[-1] - lower_line[-1]
        if range_start <= 0 or range_end < 0:
            counts['rejected_convergence_invalid_range'] += 1
            continue
        compression = (range_start - range_end) / range_start
        compression_values.append(compression)
        if compression < min_convergence_pct:
            counts['rejected_compression'] += 1
            continue

        flat_threshold = 0.1 * atr_i / window
        is_ascending = abs(slmax) < flat_threshold and slmin > flat_threshold
        is_descending = slmax < -flat_threshold and abs(slmin) < flat_threshold
        is_symmetric = slmax < -flat_threshold and slmin > flat_threshold

        if not (is_ascending or is_descending or is_symmetric):
            counts['rejected_no_classification'] += 1
            continue

        tol = 0.1 * atr_i
        cr = containment_ratio(highs, lows, upper_line, lower_line, tolerance=tol)
        containment_values.append(cr)
        if cr < 0.80:
            counts['rejected_containment'] += 1
            continue

        current_high = df["High"].iloc[i]
        current_low = df["Low"].iloc[i]
        recent_high = highs[-3:].max()
        recent_low = lows[-3:].min()
        range_breakout_up = current_high > recent_high + 0.3 * atr_i
        range_breakout_down = current_low < recent_low - 0.3 * atr_i

        current_close = df["Close"].iloc[i]
        upper_at_current = np.polyval(high_coeffs, window)
        lower_at_current = np.polyval(low_coeffs, window)
        breakout_direction = None
        if range_breakout_up and current_close > upper_at_current:
            breakout_direction = "up"
        elif range_breakout_down and current_close < lower_at_current:
            breakout_direction = "down"

        if breakout_direction is not None:
            counts['accepted_breakout'] += 1
            bars_since_last = 0
            continue

        if is_descending and abs(current_close - upper_at_current) < 0.3 * atr_i:
            counts['accepted_upper_test'] += 1
            bars_since_last = 0
            continue

        counts['rejected_no_confirmed_breakout'] += 1

    return counts, r_values, compression_values, containment_values

In [16]:
counts, r_vals, comp_vals, cont_vals = triangle_rejection_funnel(
    df_yf, window=25, pivot_order=3, min_r=0.85
)

print("=== TRIANGLE REJECTION FUNNEL (original config: window=25, pivot_order=3) ===")
for k, v in counts.items():
    print(f"  {k}: {v}")

print()
r_max_vals = [r[0] for r in r_vals]
r_min_vals = [r[1] for r in r_vals]
print(f"|r_upper|: mean={np.mean(r_max_vals):.3f}, % passing 0.85: {100*np.mean(np.array(r_max_vals)>=0.85):.1f}%")
print(f"|r_lower|: mean={np.mean(r_min_vals):.3f}, % passing 0.85: {100*np.mean(np.array(r_min_vals)>=0.85):.1f}%")
print(f"Compression: % passing 0.05: {100*np.mean(np.array(comp_vals)>=0.05):.1f}%")
print(f"Containment: % passing 0.80: {100*np.mean(np.array(cont_vals)>=0.80):.1f}%")

=== TRIANGLE REJECTION FUNNEL (original config: window=25, pivot_order=3) ===
  candidates_evaluated: 3879
  rejected_atr_invalid: 0
  rejected_cooldown: 120
  rejected_insufficient_pivots_per_side: 2362
  rejected_insufficient_total_pivots: 0
  rejected_r_threshold: 325
  rejected_convergence_invalid_range: 232
  rejected_compression: 552
  rejected_no_classification: 349
  rejected_containment: 17
  rejected_no_confirmed_breakout: 30
  accepted_breakout: 11
  accepted_upper_test: 1

|r_upper|: mean=nan, % passing 0.85: 91.7%
|r_lower|: mean=0.935, % passing 0.85: 85.2%
Compression: % passing 0.05: 42.5%
Containment: % passing 0.80: 71.2%


## Key finding

The dominant bottleneck is **insufficient pivot detection** (~61% of all
candidates rejected here), not the `|r| ≥ 0.85` fit-quality threshold as
might be assumed — 85–92% of candidates that reach the |r| check already
pass it. This directs the ablation toward `window` and `pivot_order`,
not the fit-quality or compression thresholds.

In [17]:
results = []
for window in [20, 25, 30, 40]:
    for pivot_order in [2, 3]:
        counts_i, _, _, _ = triangle_rejection_funnel(
            df_yf, window=window, pivot_order=pivot_order
        )
        results.append({
            'window': window,
            'pivot_order': pivot_order,
            'accepted': counts_i['accepted_breakout'] + counts_i['accepted_upper_test'],
            'rejected_pivots': counts_i['rejected_insufficient_pivots_per_side'],
            'rejected_compression': counts_i['rejected_compression'],
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 window  pivot_order  accepted  rejected_pivots  rejected_compression
     20            2        22             1576                   708
     20            3         9             3344                   266
     25            2        11              563                   829
     25            3        12             2362                   552
     30            2        10              165                   692
     30            3        14             1398                   728
     40            2         0                6                   504
     40            3         4              266                   749


In [18]:
_, tri_details_new = detect_triangle_pattern(
    df_yf.copy(), window=20, pivot_order=2, return_details=True
)

tri_scorecard_new = pd.DataFrame([
    {
        'date': d['event_date'], 'type': d['pattern_type'],
        'containment': d.get('containment_ratio'),
        'r_upper': d.get('r_upper'), 'r_lower': d.get('r_lower'),
    }
    for d in tri_details_new
])

print(f"New config (window=20, pivot_order=2): {len(tri_details_new)} triangles")
print(f"Mean containment: {tri_scorecard_new['containment'].mean():.3f} (was 0.873)")
print(f"Mean |r_upper|: {tri_scorecard_new['r_upper'].mean():.3f} (was 0.978)")
print(f"Mean |r_lower|: {tri_scorecard_new['r_lower'].mean():.3f} (was 0.996)")

New config (window=20, pivot_order=2): 34 triangles
Mean containment: 0.906 (was 0.873)
Mean |r_upper|: 0.992 (was 0.978)
Mean |r_lower|: 0.981 (was 0.996)


## 3. Secondary Finding: Triangle-Type Classification Bug

The window/pivot_order fix alone produced 22 detections, but **all 22 were
classified as "symmetric_triangle"** — ascending and descending types had
effectively vanished. Investigation revealed the classification threshold
was a *fixed*, tiny ATR-scaled value, while real observed slopes were
10–50x larger — making the "is this slope flat?" test almost impossible
to satisfy in either direction. This means ascending/descending triangles
were structurally unreachable, independent of real market behavior.

In [19]:
def triangle_type_diagnostic(df, window=20, pivot_order=2, min_r=0.85, min_convergence_pct=0.05):
    atr = compute_atr(df, window=14)
    slope_pairs = []
    for i in range(window, len(df)):
        atr_i = atr.iloc[i]
        if pd.isna(atr_i) or atr_i <= 0:
            continue
        window_slice = df.iloc[i - window: i]
        highs = window_slice["High"].values
        lows = window_slice["Low"].values
        sh_idx = find_swing_highs(highs, order=pivot_order)
        sl_idx = find_swing_lows(lows, order=pivot_order)
        if len(sh_idx) < 2 or len(sl_idx) < 2:
            continue
        sh_x, sh_y = np.array(sh_idx, dtype=float), highs[sh_idx]
        sl_x, sl_y = np.array(sl_idx, dtype=float), lows[sl_idx]
        slmax, _, rmax, _, _ = linregress(sh_x, sh_y)
        slmin, _, rmin, _, _ = linregress(sl_x, sl_y)
        if abs(rmax) < min_r or abs(rmin) < min_r:
            continue
        flat_threshold = 0.1 * atr_i / window
        slope_pairs.append((slmax, slmin, flat_threshold))
    return slope_pairs

pairs = triangle_type_diagnostic(df_yf)

ascending_current = sum(1 for slmax, slmin, ft in pairs if abs(slmax) < ft and slmin > ft)
descending_current = sum(1 for slmax, slmin, ft in pairs if slmax < -ft and abs(slmin) < ft)
symmetric_current = sum(1 for slmax, slmin, ft in pairs if slmax < -ft and slmin > ft)
none_current = len(pairs) - ascending_current - descending_current - symmetric_current
print(f"Fixed threshold — ascending: {ascending_current}, descending: {descending_current}, "
      f"symmetric: {symmetric_current}, none: {none_current}")

for mult in [0.15, 0.25, 0.35]:
    asc, desc, sym, none = 0, 0, 0, 0
    for slmax, slmin, ft in pairs:
        scale = max(abs(slmax), abs(slmin), 1e-9)
        rel_thresh = mult * scale
        is_asc = abs(slmax) < rel_thresh and slmin > rel_thresh
        is_desc = slmax < -rel_thresh and abs(slmin) < rel_thresh
        is_sym = slmax < -rel_thresh and slmin > rel_thresh
        if is_asc: asc += 1
        elif is_desc: desc += 1
        elif is_sym: sym += 1
        else: none += 1
    print(f"Relative threshold (mult={mult}) — ascending: {asc}, descending: {desc}, "
          f"symmetric: {sym}, none: {none}")

Fixed threshold — ascending: 9, descending: 0, symmetric: 193, none: 1521
Relative threshold (mult=0.15) — ascending: 76, descending: 27, symmetric: 159, none: 1461
Relative threshold (mult=0.25) — ascending: 127, descending: 51, symmetric: 125, none: 1420
Relative threshold (mult=0.35) — ascending: 197, descending: 80, symmetric: 85, none: 1361


## 4. Final Fix Applied

Two changes were made to `src/patterns/triangles.py`:
1. Defaults changed from `window=25, pivot_order=3` to `window=20, pivot_order=2`
2. Classification threshold changed from a fixed ATR-scaled value to a
   relative threshold (`flat_threshold_mult=0.25`), scaled to the steeper
   of the two observed slopes.

In [20]:
import pandas as pd
import numpy as np

_, tri_details_final = detect_triangle_pattern(df_yf.copy(), return_details=True)

type_counts_final = pd.Series([d['pattern_type'] for d in tri_details_final]).value_counts()
print(f"Total triangles (final, both fixes applied): {len(tri_details_final)}")
print(type_counts_final)

final_scorecard = pd.DataFrame([
    {'date': d['event_date'], 'type': d['pattern_type'],
     'containment': d.get('containment_ratio'),
     'r_upper': d.get('r_upper'), 'r_lower': d.get('r_lower')}
    for d in tri_details_final
])
print()
print(f"Mean containment: {final_scorecard['containment'].mean():.3f}")
print(f"Mean |r_upper|: {final_scorecard['r_upper'].mean():.3f}")
print(f"Mean |r_lower|: {final_scorecard['r_lower'].mean():.3f}")

Total triangles (final, both fixes applied): 34
symmetric_triangle          15
ascending_triangle          13
desc_triangle_upper_test     4
descending_triangle          2
Name: count, dtype: int64

Mean containment: 0.906
Mean |r_upper|: 0.992
Mean |r_lower|: 0.981


In [21]:
new_ascending = [d for d in tri_details_final if d['pattern_type'] == 'ascending_triangle']
new_descending = [d for d in tri_details_final if d['pattern_type'] == 'descending_triangle']
print(f"Ascending: {len(new_ascending)}, Descending: {len(new_descending)}")

Ascending: 13, Descending: 2


## Conclusion & Next Steps

**Before → After:**
| Metric | Before | After |
|---|---:|---:|
| Total triangles | 12 | 34 (+183%)|
| Type diversity | 10 sym / 1 asc / 1 desc-test | 15 symmetric / 13 ascending / 4 desc-test / 2 descending|
| Mean containment | 0.873 | 0.906 |
| Mean \|r_upper\| | 0.978 | 0.992 |

**Two evidence-based fixes applied:**
1. Window/pivot_order tuning, directed by rejection-funnel diagnosis
   (bottleneck was pivot detection, not fit quality)
2. Classification threshold changed from fixed to relative, fixing a
   structural bug that made ascending/descending triangles nearly
   unreachable regardless of true market behavior

**Next steps:** apply the same rejection-funnel methodology to the
support/resistance detector's support/resistance imbalance (3 vs 34).